In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!apt-get install poppler-utils
!pip install pdf2image

from pdf2image import convert_from_path
from pathlib import Path

pdf_dir = Path("/content/drive/MyDrive/dataset/attestations_urssaf")
img_dir = Path("/content/drive/MyDrive/dataset/images_clean")
img_dir.mkdir(exist_ok=True)

for pdf_file in pdf_dir.glob("*.pdf"):
    images = convert_from_path(pdf_file, dpi=200)

    for i, img in enumerate(images):
        img.save(img_dir / f"{pdf_file.stem}_{i}.png")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 1s (283 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...


In [7]:
import cv2
import numpy as np
import random
from pathlib import Path

input_dir = Path("/content/drive/MyDrive/dataset/images_clean")
output_dir = Path("/content/drive/MyDrive/dataset/images_noisy")
output_dir.mkdir(exist_ok=True)

def add_noise(image):
    noise = np.random.normal(0, 25, image.shape).astype(np.uint8)
    return cv2.add(image, noise)

def add_blur(image):
    k = random.choice([3,5])
    return cv2.GaussianBlur(image, (k,k), 0)

def add_motion_blur(image):
    size = random.randint(5, 15)
    kernel = np.zeros((size, size))
    kernel[int((size-1)/2), :] = np.ones(size)
    kernel = kernel / size
    return cv2.filter2D(image, -1, kernel)

def rotate(image):
    angle = random.uniform(-5, 5)
    h, w = image.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1)
    return cv2.warpAffine(image, M, (w, h), borderMode=cv2.BORDER_REPLICATE)

def perspective_transform(image):
    h, w = image.shape[:2]

    shift = 20
    pts1 = np.float32([[0,0],[w,0],[0,h],[w,h]])
    pts2 = np.float32([
        [random.randint(0,shift), random.randint(0,shift)],
        [w-random.randint(0,shift), random.randint(0,shift)],
        [random.randint(0,shift), h-random.randint(0,shift)],
        [w-random.randint(0,shift), h-random.randint(0,shift)]
    ])

    M = cv2.getPerspectiveTransform(pts1, pts2)
    return cv2.warpPerspective(image, M, (w, h))

def jpeg_compression(image):
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), random.randint(30, 70)]
    _, encimg = cv2.imencode('.jpg', image, encode_param)
    return cv2.imdecode(encimg, 1)

def change_brightness(image):
    factor = random.uniform(0.6, 1.4)
    return np.clip(image * factor, 0, 255).astype(np.uint8)

def degrade(image):
    if random.random() < 0.7:
        image = rotate(image)

    if random.random() < 0.5:
        image = perspective_transform(image)

    if random.random() < 0.6:
        image = add_blur(image)

    if random.random() < 0.5:
        image = add_motion_blur(image)

    if random.random() < 0.7:
        image = add_noise(image)

    if random.random() < 0.8:
        image = change_brightness(image)

    if random.random() < 0.9:
        image = jpeg_compression(image)

    return image

In [8]:
for img_path in input_dir.glob("*.png"):
    image = cv2.imread(str(img_path))

    for i in range(3):  # 3 versions différentes
        degraded = degrade(image)
        output_path = output_dir / f"{img_path.stem}_noisy_{i}.jpg"
        cv2.imwrite(str(output_path), degraded)